# SmoothRoute: study a recorded ride

This notebook loads a CSV downloaded from the recorder page and helps you answer:

1. What does a real bump look like in the data?
2. How noisy is a smooth road?
3. Does speed change how big a bump looks?
4. Which threshold catches your marked bumps without too many false alarms?

**How to use:** put your CSV in the `data/` folder, set `CSV_PATH` in the first code cell, then choose **Run, Run All Cells**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---------------- change these ----------------
CSV_PATH = Path("../data/my_ride.csv")   # the file you downloaded from the app
MIN_SPEED_MS = 3.0    # readings slower than this are ignored (3 m/s is about 11 km/h). For a walking test use 0.5
PRE_WINDOW_S = 2.0    # a mark is tapped AFTER the bump, so we look this many seconds BEFORE the tap...
POST_WINDOW_S = 0.5   # ...and this many seconds after it
PAUSE_S = 1.0         # after a detected bump, ignore further bumps for this long
# ------------------------------------------------

MARK_COLORS = {"speed_breaker": "tab:orange", "pothole": "tab:red", "rough_patch": "tab:purple"}
plt.rcParams["figure.figsize"] = (12, 4)

## 1. Load the file
Lines starting with `#` are your session notes (phone, mounting, vehicle, road). Rows with an `event` are the bumps you marked by hand; all other rows are sensor readings.

In [ ]:
raw = pd.read_csv(CSV_PATH, comment="#")
raw["event"] = raw["event"].fillna("")

marks = raw[raw["event"] != ""].copy()
data = raw[raw["event"] == ""].dropna(subset=["strength"]).reset_index(drop=True)
if data.empty:
    raise ValueError("No motion readings in this file. Record on a phone (laptops have no motion sensor).")

t0 = data["time_ms"].iloc[0]
data["t"] = (data["time_ms"] - t0) / 1000      # seconds since the first reading
marks["t"] = (marks["time_ms"] - t0) / 1000

In [ ]:
rate = 1 / data["t"].diff().median()
gravity = bool(data["gravity_included"].mode()[0])
speed_kmh = data["speed_mps"] * 3.6

print(f"Readings: {len(data):,}    Duration: {data['t'].iloc[-1]:.0f} s    Sensor rate: {rate:.0f} per second")
print("Values include gravity (a still phone reads about 9.8)" if gravity else "Values do NOT include gravity (a still phone reads about 0)")
print(f"Speed: average {speed_kmh.mean():.1f} km/h, max {speed_kmh.max():.1f} km/h")
print("Marks:", marks["event"].value_counts().to_dict() if len(marks) else "none")

## 2. Remove the normal background shaking
A moving vehicle always vibrates a little, and with gravity included the reading sits near 9.8. We estimate that slow background with a **rolling median over about one second** and measure how far each reading is from it. This distance is called the **jolt**. (In the research papers this step is a *high-pass filter*.)

In [ ]:
window = max(5, int(round(rate)))     # about 1 second of readings
baseline = data["strength"].rolling(window, center=True, min_periods=1).median()
data["jolt"] = (data["strength"] - baseline).abs()

## 3. The whole ride
Top: raw strength and its baseline. Middle: the jolt. Bottom: speed. Each coloured vertical line is a bump you marked.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(data["t"], data["strength"], lw=0.6, label="strength")
axes[0].plot(data["t"], baseline, lw=1.2, color="black", label="baseline (slow average)")
axes[0].set_ylabel("strength (m/s²)")
axes[0].legend(loc="upper right")
axes[1].plot(data["t"], data["jolt"], lw=0.6, color="tab:blue")
axes[1].set_ylabel("jolt (distance from baseline)")
axes[2].plot(data["t"], speed_kmh, color="tab:green")
axes[2].set_ylabel("speed (km/h)")
axes[2].set_xlabel("time since start (s)")
for _, m in marks.iterrows():
    for ax in axes:
        ax.axvline(m["t"], color=MARK_COLORS.get(m["event"], "gray"), alpha=0.7, lw=1.5)
handles = [plt.Line2D([0], [0], color=c, lw=2, label=k) for k, c in MARK_COLORS.items()]
axes[1].legend(handles=handles, loc="upper right", title="your marks")
plt.tight_layout()
plt.show()

### Zoom in on one mark
Notice the spike comes a little **before** the coloured line, because the tap happens after you cross the bump.

In [ ]:
def zoom(center_s, before=6, after=3):
    d = data[(data["t"] >= center_s - before) & (data["t"] <= center_s + after)]
    plt.figure(figsize=(12, 3.5))
    plt.plot(d["t"], d["jolt"], lw=0.8)
    near = marks[(marks["t"] >= center_s - before) & (marks["t"] <= center_s + after)]
    for _, m in near.iterrows():
        plt.axvline(m["t"], color=MARK_COLORS.get(m["event"], "gray"), lw=2, label=f"mark: {m['event']}")
    plt.xlabel("time since start (s)")
    plt.ylabel("jolt")
    plt.legend()
    plt.show()

if len(marks):
    zoom(marks["t"].iloc[0])    # change the number to look at another mark, e.g. zoom(marks["t"].iloc[2])

## 4. What does a real bump look like?
For each mark, find the biggest jolt in the few seconds **before the tap**. `tap_delay_s` is how long after the bump you tapped.

In [ ]:
rows = []
for _, m in marks.iterrows():
    w = data[(data["t"] >= m["t"] - PRE_WINDOW_S) & (data["t"] <= m["t"] + POST_WINDOW_S)]
    if w.empty:
        continue
    i = w["jolt"].idxmax()
    rows.append({
        "type": m["event"],
        "mark_at_s": round(m["t"], 1),
        "peak_jolt": round(w.loc[i, "jolt"], 2),
        "peak_at_s": round(w.loc[i, "t"], 2),
        "tap_delay_s": round(m["t"] - w.loc[i, "t"], 2),
        "speed_kmh": round(w.loc[i, "speed_mps"] * 3.6, 1),
    })
peaks = pd.DataFrame(rows)
display(peaks)

## 5. How noisy is a smooth road?
We take only readings where you were moving (above `MIN_SPEED_MS`) and **not** within a few seconds of a mark. Their jolt values show the normal background level a bump has to rise above.

In [ ]:
near_mark = np.zeros(len(data), dtype=bool)
for mt in marks["t"]:
    near_mark |= ((data["t"] > mt - 3.0) & (data["t"] < mt + 1.0)).to_numpy()
moving = (data["speed_mps"] >= MIN_SPEED_MS).to_numpy()      # missing speed counts as not moving
smooth = data[moving & ~near_mark].copy()

if smooth.empty:
    print("No smooth-road readings above the minimum speed. Lower MIN_SPEED_MS (for example 0.5 for a walk).")
else:
    print(f"Smooth-road readings: {len(smooth):,}")
    display(smooth["jolt"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(3))

In [ ]:
if not smooth.empty:
    plt.figure(figsize=(12, 3.5))
    plt.hist(smooth["jolt"], bins=80, color="lightgray", log=True)
    for _, p in peaks.iterrows():
        plt.axvline(p["peak_jolt"], color=MARK_COLORS.get(p["type"], "gray"), lw=2)
    plt.xlabel("jolt")
    plt.ylabel("number of readings (log scale)")
    plt.title("Grey = smooth road. Coloured lines = the peak next to each of your marks")
    plt.show()

## 6. Does speed change how big a bump looks?
If the points rise to the right, the same bump feels bigger when you go faster. That is why SmoothRoute divides the jolt by speed (**speed normalisation**).

In [ ]:
if len(peaks) >= 3 and peaks["speed_kmh"].notna().all():
    plt.figure(figsize=(6, 4))
    for kind, grp in peaks.groupby("type"):
        plt.scatter(grp["speed_kmh"], grp["peak_jolt"], label=kind, color=MARK_COLORS.get(kind, "gray"))
    plt.xlabel("speed (km/h)")
    plt.ylabel("peak jolt")
    plt.title("Faster = bigger jolt?")
    plt.legend()
    plt.show()
else:
    print("Need at least 3 marks with a speed value to draw this plot.")

## 7. Try thresholds (a preview of automatic detection)
A reading counts as a **bump** when its value is above the threshold, you were moving fast enough, and at least `PAUSE_S` seconds have passed since the last bump. A detection counts as **found** if it falls in the window around one of your marks, otherwise it is a **false alarm**.

We test two measures: the plain `jolt`, and `jolt_per_speed` (jolt divided by speed). Thresholds are chosen from the smooth-road data, so a higher percentile means a stricter threshold.

In [ ]:
data["jolt_per_speed"] = data["jolt"] / data["speed_mps"].clip(lower=MIN_SPEED_MS)
smooth["jolt_per_speed"] = data.loc[smooth.index, "jolt_per_speed"]

def detect(column, threshold):
    d = data[data["speed_mps"] >= MIN_SPEED_MS]
    hits, last = [], -1e9
    for t, v in zip(d["t"].to_numpy(), d[column].to_numpy()):
        if v > threshold and t - last >= PAUSE_S:
            hits.append(t)
            last = t
    return np.array(hits)

def score(hits):
    windows = [(m - PRE_WINDOW_S, m + POST_WINDOW_S) for m in marks["t"]]
    found = sum(any(lo <= h <= hi for h in hits) for lo, hi in windows)
    false_alarms = sum(not any(lo <= h <= hi for lo, hi in windows) for h in hits)
    return found, false_alarms

if smooth.empty or marks.empty:
    print("Need marks and smooth-road readings to compare thresholds.")
else:
    results = []
    for column in ["jolt", "jolt_per_speed"]:
        for q in [0.99, 0.995, 0.999, 0.9995, 0.9999]:
            thr = smooth[column].quantile(q)
            found, false_alarms = score(detect(column, thr))
            results.append({
                "measure": column,
                "threshold": round(thr, 3),
                "smooth_road_percentile": q,
                "marks_found": f"{found}/{len(marks)}",
                "detection_rate": round(found / len(marks), 2),
                "false_alarms": false_alarms,
            })
    display(pd.DataFrame(results))

## What to write down
* The **typical peak jolt** for each bump type, and the **smooth-road level** (the 99th percentile).
* The `tap_delay_s` you usually see (the average delay between a bump and your tap).
* The threshold that finds most of your marks with few false alarms, **for both measures**. If `jolt_per_speed` does better than `jolt`, that is your evidence that speed normalisation helps.
* Your phone model, how it was mounted, and the speeds you rode at.

Use several rides, at different speeds, before trusting a threshold. These numbers become the starting values for automatic detection in the app.